In [1]:
!pip install tensorflow

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from sklearn.utils import resample

# For image detection
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image

from google.colab import files

In [7]:
from sklearn.datasets import make_classification
import pandas as pd

# Create synthetic fraud-like dataset
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.95, 0.05],  # imbalanced like fraud
    random_state=42
)

data = pd.DataFrame(X)
data['Class'] = y

data.head()

,0,1,2,3,4,5,6,7,8,9,Class
0,-1.030931,1.391626,0.547274,0.928932,-1.738880,1.250002,1.332551,1.578256,2.124722,-0.318434,0
1,-1.930636,-0.406752,3.061308,1.083145,-2.934634,-1.355054,-1.794192,-0.873019,7.065968,2.110308,0
2,-0.558987,0.299849,1.527071,0.360442,-1.360209,1.100793,-0.755951,1.331933,2.041105,-0.824404,0
3,-1.350289,-2.046078,-0.614264,0.126459,-0.783923,5.895026,-0.915477,-3.184768,-0.399260,-3.920960,0
4,-0.275754,-0.728495,0.027727,-0.660834,-1.928161,3.544945,1.446944,-1.111662,0.313766,-2.376528,0


In [8]:
print(data['Class'].value_counts())

Class
0    944
1     56
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

X = data.drop('Class', axis=1)
y = data['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [11]:
from sklearn.metrics import classification_report, accuracy_score

# Logistic
y_pred_lr = lr.predict(X_test)
print("🔹 Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# Random Forest
y_pred_rf = rf.predict(X_test)
print("\n🔹 Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

🔹 Logistic Regression
Accuracy: 0.945
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       189
           1       0.50      0.18      0.27        11

    accuracy                           0.94       200
   macro avg       0.73      0.59      0.62       200
weighted avg       0.93      0.94      0.93       200


🔹 Random Forest
Accuracy: 0.965
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       189
           1       1.00      0.36      0.53        11

    accuracy                           0.96       200
   macro avg       0.98      0.68      0.76       200
weighted avg       0.97      0.96      0.96       200



In [12]:
import numpy as np

def predict_transaction(input_data):
    input_data = np.array(input_data).reshape(1, -1)
    result = rf.predict(input_data)[0]

    if result == 1:
        return "🚨 Fraud Transaction"
    else:
        return "✅ Legitimate Transaction"

In [13]:
# Example input (10 features because we created 10 features dataset)
sample_input = [0.5, -1.2, 0.3, 1.5, -0.7, 0.8, 0.2, -0.3, 1.1, 200]

result = predict_transaction(sample_input)
print(result)

✅ Legitimate Transaction


In [14]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from google.colab import files
import numpy as np

# Load pretrained model
img_model = MobileNetV2(weights='imagenet')

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [30]:
uploaded = files.upload()

Saving Blue credit card close-up shot.png to Blue credit card close-up shot.png


In [31]:
img_path = list(uploaded.keys())[0]

img = image.load_img(img_path, target_size=(224, 224))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = preprocess_input(img_array)

In [32]:
preds = img_model.predict(img_array)
decoded = decode_predictions(preds, top=3)[0]

print("Top Predictions:", decoded)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
Top Predictions: [('n04116512', 'rubber_eraser', np.float32(0.20174281)), ('n03929660', 'pick', np.float32(0.14352272)), ('n03871628', 'packet', np.float32(0.09935178))]


In [34]:
labels = [item[1] for item in decoded]

print("Detected labels:", labels)

# More flexible check
card_keywords = ["credit_card", "card", "wallet", "purse"]

if any(word in label for label in labels for word in card_keywords):
    print("✅ This looks like a Credit Card")
else:
    print("❌ It's NOT a Credit Card")

Detected labels: ['rubber_eraser', 'pick', 'packet']
❌ It's NOT a Credit Card
